In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig


# 08 Drawdown Diagnostics
Diagnose the Module 07 portfolio drawdown and realized trade outcomes.


In [ ]:
cfg = ResearchConfig()


In [ ]:
trades = pd.read_parquet("trades.parquet")
equity = pd.read_parquet("equity_curve.parquet")
running_peak = equity.equity.cummax().clip(lower=cfg.initial_capital)
drawdown = equity.equity / running_peak - 1
trough = drawdown.idxmin()
peak = equity.loc[:trough, "equity"].idxmax()

drawdown.to_frame("drawdown").to_parquet("drawdown_series.parquet")
pd.to_pickle({"peak_date": peak, "trough_date": trough, "max_drawdown": drawdown.min()}, "drawdown_episode.pkl")
display(pd.Series({"peak_date": peak, "trough_date": trough, "max_drawdown": drawdown.min()}))
drawdown.plot(figsize=(11, 3), title="Portfolio drawdown")
plt.show()


In [ ]:
daily_changes = equity.equity.diff().to_frame("equity_change")
display(daily_changes.nsmallest(10, "equity_change"))

if len(trades):
    losses_by_exit = trades.groupby("exit_date").pnl.sum().to_frame("realized_pnl")
    losses_by_exit.to_parquet("pnl_by_exit_date.parquet")
    pair_performance = trades.groupby("pair").agg(
        n_trades=("pnl", "size"),
        total_pnl=("pnl", "sum"),
        mean_trade_return=("trade_return", "mean"),
        win_rate=("pnl", lambda x: (x > 0).mean()),
    )
    overlapping = trades.loc[(trades.entry_date <= trough) & (trades.exit_date >= peak)]
    pair_performance.to_parquet("pair_performance.parquet")
    overlapping.to_parquet("drawdown_overlapping_trades.parquet")
    display(pair_performance.sort_values("total_pnl").head(15))

equity[["n_open_positions"]].plot(figsize=(10, 3), title="Number of open pairs")
plt.show()
